# Глава 3. Датасет с временными рядами
## CWRU Bearing Dataset (Case Western Reserve University)
**Кейс № 62:** диагностика подшипников с помощью гибридной CNN-MLP модели  
**Задача:** многоклассовая классификация вибрационных сигналов (5 классов)  
**Источник:** https://engineering.case.edu/bearingdatacenter  
**Датасет:** файлы формата `.mat`, частота дискретизации 12 000 Гц, 5 классов состояния подшипника

---
## Этап 1. Загрузка и первичное знакомство с данными
Загружаем пять `.mat`-файлов — по одному на каждый класс состояния подшипника.
Файлы загружаются вручную через диалог `files.upload()` в следующем порядке:
1. Норма
2. Дефект внутреннего кольца
3. Дефект шарика
4. Дефект внешнего кольца (Centered)
5. Дефект внешнего кольца (Orthogonal)

По результатам этапа выводятся: первые 20 значений каждого сигнала, длина ряда, тип данных, базовые статистики и сводная таблица по всем пяти файлам.

In [ ]:
# Этап 1 Загрузка и первичное знакомство

import scipy.io as sio
import pandas as pd
import numpy as np
from google.colab import files

# Список для хранения данных
file_paths = []
classes = [
    'Норма',
    'Дефект внутреннего кольца',
    'Дефект шарика',
    'Дефект внешнего кольца (Centered)',
    'Дефект внешнего кольца (Orthogonal)'
]

# Загрузка 5 файлов
print("Загрузите 5 файлов в следующем порядке:")
for i, cls in enumerate(classes, 1):
    print(f"{i}. {cls}")
for i in range(5):
    print(f"\n--- Загрузите файл {i+1} ---")
    uploaded = files.upload()
    file_paths.append(list(uploaded.keys())[0])

print("ПОДРОБНЫЙ АНАЛИЗ КАЖДОГО ФАЙЛА")

signals = []
signal_keys = []

for idx, (file, cls) in enumerate(zip(file_paths, classes)):
    print(f"\n--- {cls} ---")
    print(f"Файл: {file}")

    mat = sio.loadmat(file)

    all_keys = [k for k in mat.keys() if not k.startswith('__')]
    print(f"Ключи в файле: {all_keys}")

    signal_key = None
    for key in all_keys:
        obj = mat[key]
        if isinstance(obj, np.ndarray) and obj.ndim == 2 and obj.shape[1] == 1:
            if 'DE_time' in key or ('X' in key and 'DE' in key):
                signal_key = key
                break
    if signal_key is None:
        for key in all_keys:
            obj = mat[key]
            if isinstance(obj, np.ndarray) and obj.size > 1000:
                signal_key = key
                break

    signal = mat[signal_key].flatten()
    signals.append(signal)
    signal_keys.append(signal_key)

    # Первые 20 значений сигнала
    print(f"\n--- ПЕРВЫЕ 20 ЗНАЧЕНИЙ СИГНАЛА (ключ '{signal_key}') ---")
    print(signal[:20])

    # Общая информация о данных
    print("\n--- ОБЩАЯ ИНФОРМАЦИЯ ---")
    print(f"Количество каналов: 1 (одномерный временной ряд)")
    print(f"Длина ряда: {len(signal)} отсчётов")
    print(f"Тип данных: {signal.dtype}")
    print(f"Размерность массива: {signal.shape}")

    # Краткая статистика
    print("\n--- СТАТИСТИЧЕСКИЕ ХАРАКТЕРИСТИКИ (предварительные) ---")
    print(f"  min = {signal.min():.6f}")
    print(f"  max = {signal.max():.6f}")
    print(f"  mean = {signal.mean():.6f}")
    print(f"  std  = {signal.std():.6f}")

    # Проверка на пропуски
    if np.isnan(signal).any():
        print(f"\nПредупреждение: в сигнале есть пропуски (NaN). Количество: {np.isnan(signal).sum()}")
    else:
        print("\nПропуски (NaN) отсутствуют.")

# Сводная таблица по всем пяти файлам
print("СВОДНАЯ ТАБЛИЦА ПО 5 ФАЙЛАМ")

summary_data = []
for cls, file, key, sig in zip(classes, file_paths, signal_keys, signals):
    summary_data.append({
        'Класс': cls,
        'Файл': file,
        'Ключ сигнала': key,
        'Длина ряда': len(sig),
        'Тип данных': str(sig.dtype),
        'Пропуски': 0,
        'Временная метка': 'отсутствует'
    })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

---
## Этап 2. Визуализация исходных данных — отдельные графики
Строятся три серии индивидуальных графиков для каждого из пяти классов:
- **Полный сигнал** — вся длительность записи, позволяет оценить общий уровень амплитуды;
- **Фрагмент 0.1 с** (1 200 отсчётов) — крупная структура ударных импульсов;
- **Фрагмент 0.05 с** (600 отсчётов) — детальная форма и частота отдельных импульсов.

Каждый график подписан и содержит название класса, подписи осей и сетку.

In [ ]:
# ============================================
# Этап 2. Визуализация исходных данных (отдельные графики, полная длительность)
# ============================================

import matplotlib.pyplot as plt
import numpy as np

fs = 12000
time_arrays = [np.arange(len(sig)) / fs for sig in signals]

plt.style.use('classic')
plt.rcParams['figure.figsize'] = (12, 4)

colors = plt.cm.tab10(np.linspace(0, 1, 5))

# 1. Полные сигналы (каждый на отдельной фигуре, вся длительность)
for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    plt.figure(figsize=(12, 4))
    plt.plot(time, sig, linewidth=0.5, color=colors[i])
    plt.title(f'Полный вибрационный сигнал: {name}')
    plt.xlabel('Время (секунды)')
    plt.ylabel('Амплитуда')
    plt.grid(alpha=0.3)
    plt.xlim(0, time[-1])   # от 0 до конца записи
    plt.tight_layout()
    # plt.savefig(f'full_{name.replace(" ", "_")}.png', dpi=150)  # опционально
    plt.show()

# 2. Фрагменты (первые 0.1 с) – отдельные фигуры
duration_short = 0.1
samples_short = int(duration_short * fs)

for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    plt.figure(figsize=(12, 4))
    plt.plot(time[:samples_short], sig[:samples_short], linewidth=0.8, color=colors[i])
    plt.title(f'Фрагмент (первые 0.1 с): {name}')
    plt.xlabel('Время (секунды)')
    plt.ylabel('Амплитуда')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# 3. Детальный анализ (первые 0.05 с) – отдельные фигуры
duration_detailed = 0.05
samples_detailed = int(duration_detailed * fs)

for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    plt.figure(figsize=(12, 4))
    plt.plot(time[:samples_detailed], sig[:samples_detailed],
             linewidth=1.2, color=colors[i], marker='o', markersize=2, markevery=20)
    plt.title(f'Детальный анализ (первые 0.05 с): {name}')
    plt.xlabel('Время (секунды)')
    plt.ylabel('Амплитуда')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Этап 2. Визуализация исходных данных — сводные графики
Те же три вида визуализации, но оформленные в виде сводных рисунков: все пять классов выводятся на одном рисунке в виде субграфиков, расположенных вертикально друг под другом. Такой формат удобен для сравнения амплитуды и характера колебаний между классами.

In [ ]:
# Этап 2. Визуализация исходных данных

import matplotlib.pyplot as plt
import numpy as np

fs = 12000

time_arrays = [np.arange(len(sig)) / fs for sig in signals]

plt.style.use('classic')
plt.rcParams['figure.figsize'] = (14, 10)

colors = plt.cm.tab10(np.linspace(0, 1, 5))

# 1. Полные сигналы
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(14, 12))
fig.suptitle('Полные вибрационные сигналы (5 состояний подшипника)', fontsize=14)

for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    duration = min(2.0, len(sig)/fs)
    samples = int(duration * fs)
    axes[i].plot(time[:samples], sig[:samples], linewidth=0.5, color=colors[i])
    axes[i].set_ylabel('Амплитуда')
    axes[i].set_title(name)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Время (секунды)')
plt.tight_layout()
plt.show()

# 2. Фрагменты сигналов (первые 0.1 секунды)
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(14, 12))
fig.suptitle('Сравнение вибрационных сигналов (первые 0.1 секунды)', fontsize=14)

duration_short = 0.1
samples_short = int(duration_short * fs)

for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    axes[i].plot(time[:samples_short], sig[:samples_short], linewidth=0.8, color=colors[i])
    axes[i].set_ylabel('Амплитуда')
    axes[i].set_title(name)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Время (секунды)')
plt.tight_layout()
plt.show()

# 3. Детальный анализ (первые 0.05 секунды)
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(14, 12))
fig.suptitle('Детальный анализ (первые 0.05 секунды) – видны ударные импульсы', fontsize=14)

duration_detailed = 0.05
samples_detailed = int(duration_detailed * fs)

for i, (sig, name, time) in enumerate(zip(signals, classes, time_arrays)):
    axes[i].plot(time[:samples_detailed], sig[:samples_detailed],
                 linewidth=1.2, color=colors[i], marker='o', markersize=2, markevery=20)
    axes[i].set_ylabel('Амплитуда')
    axes[i].set_title(name)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Время (секунды)')
plt.tight_layout()
plt.show()

---
## Этап 3. Статистический анализ
Для каждого из пяти вибрационных сигналов рассчитываются описательные статистики: количество отсчётов, среднее, стандартное отклонение, минимум, квартили Q1/Q2/Q3, максимум, асимметрия (skewness) и эксцесс (kurtosis).

Результаты представлены в виде сводной таблицы. Дополнительно выполняется анализ симметричности распределения (сравнение среднего и медианы) и фиксируется частота дискретизации 12 000 Гц.

In [ ]:
# Этап 3. Статистический анализ

import numpy as np
import pandas as pd
from scipy import stats

fs = 12000

# Функция для расчёта статистик
def compute_stats(signal):
    q1 = np.percentile(signal, 25)
    q2 = np.percentile(signal, 50)
    q3 = np.percentile(signal, 75)
    skewness = stats.skew(signal)
    kurt = stats.kurtosis(signal)
    return {
        'count': len(signal),
        'mean': np.mean(signal),
        'std': np.std(signal),
        'min': np.min(signal),
        'q1': q1,
        'median': q2,
        'q3': q3,
        'max': np.max(signal),
        'skewness': skewness,
        'kurtosis': kurt
    }

stats_list = []
for name, sig in zip(classes, signals):
    stats_list.append(compute_stats(sig))

df_stats = pd.DataFrame(stats_list)
df_stats.index = classes
df_stats = df_stats.round(6)

print("\nТАБЛИЦА 1. СТАТИСТИЧЕСКИЕ ХАРАКТЕРИСТИКИ СИГНАЛОВ")
print("="*60)
print(df_stats.to_string())

# Частота дискретизации
print(f"Частота дискретизации: {fs} Гц (12 кГц)")
print(f"Интервал между отсчётами: {1/fs:.6f} секунд")

# Анализ разброса и симметричности
print("АНАЛИЗ РАЗБРОСА И СИММЕТРИЧНОСТИ")
for name, sig in zip(classes, signals):
    mean_val = np.mean(sig)
    median_val = np.percentile(sig, 50)
    std_val = np.std(sig)
    print(f"\n{name}:")
    print(f"  Среднее = {mean_val:.6f}, Медиана = {median_val:.6f}, разница = {abs(mean_val-median_val):.6f}")
    print(f"  Стандартное отклонение = {std_val:.6f}")
    if abs(mean_val-median_val) < 0.01 * abs(mean_val):
        print(" Распределение близко к симметричному")
    else:
        print(" Распределение асимметрично")

---
## Этап 4. Анализ пропусков и выбросов
Выполняются четыре шага:
1. **Проверка пропущенных значений (NaN)** — подсчёт и вывод доли пропусков для каждого сигнала.
2. **Выбросы по правилу трёх сигм** — для каждого сигнала определяются границы [μ − 3σ; μ + 3σ] и подсчитывается количество значений за их пределами.
3. **Диаграммы размаха (box plot)** — визуализация выбросов по всем пяти классам на одном рисунке.
4. **Фрагменты сигналов с выделенными выбросами** — первые 0.1 с каждого сигнала, точки-выбросы отмечены красным цветом.

In [ ]:
# Этап 4. Анализ пропусков и выбросов

import numpy as np
import matplotlib.pyplot as plt

# 1. Проверка пропущенных значений
print("\n1. ПРОВЕРКА ПРОПУЩЕННЫХ ЗНАЧЕНИЙ (NaN)")
for name, sig in zip(classes, signals):
    nan_count = np.isnan(sig).sum()
    nan_percent = nan_count / len(sig) * 100
    print(f"{name}:")
    print(f"   Пропусков: {nan_count} из {len(sig)} ({nan_percent:.4f}%)")
    if nan_count == 0:
        print(" Пропуски отсутствуют.")

# 2. Выбросы по правилу трех сигм
print("\n2. ВЫБРОСЫ ПО ПРАВИЛУ ТРЁХ СИГМ")
outliers_stats = []

for name, sig in zip(classes, signals):
    mean = np.mean(sig)
    std = np.std(sig)
    lower = mean - 3 * std
    upper = mean + 3 * std
    outliers = sig[(sig < lower) | (sig > upper)]
    count_out = len(outliers)
    percent_out = count_out / len(sig) * 100
    outliers_stats.append((name, count_out, percent_out, lower, upper))
    print(f"{name}:")
    print(f"   Границы: [{lower:.4f}, {upper:.4f}]")
    print(f"   Выбросов: {count_out} ({percent_out:.3f}%)")
    if percent_out < 1:
        print(" Доля выбросов незначительна.")
    else:
        print(" Имеется значительное количество выбросов (характерно для дефектов).")

# 3. Визуализация: диаграммы размаха (boxplot) всех каналов
print("\n3. ВИЗУАЛИЗАЦИЯ ВЫБРОСОВ (BOXPLOT)")
plt.figure(figsize=(12, 8))
data_to_plot = signals
labels = classes

bp = plt.boxplot(data_to_plot, labels=labels, patch_artist=True, notch=False, vert=True)

# Раскрашиваем ящики
colors = plt.cm.tab10(np.linspace(0, 1, len(signals)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.title('Диаграммы размаха для выявления выбросов (5 состояний)', fontsize=14)
plt.ylabel('Амплитуда вибрации')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 4. Детальная визуализация выбросов на временных рядах
print("\n4. ФРАГМЕНТЫ СИГНАЛОВ С ВЫБРОСАМИ (первые 0.1 секунды)")
fs = 12000
samples_short = 1200  # 0.1 секунды

fig, axes = plt.subplots(5, 1, sharex=True, figsize=(12, 10))
fig.suptitle('Фрагменты сигналов с выделенными выбросами (красные точки)', fontsize=14)

for i, (name, sig) in enumerate(zip(classes, signals)):
    mean = np.mean(sig)
    std = np.std(sig)
    lower = mean - 3 * std
    upper = mean + 3 * std
    time = np.arange(min(samples_short, len(sig))) / fs
    axes[i].plot(time, sig[:samples_short], linewidth=0.8, color='gray')
    outliers_mask = (sig[:samples_short] < lower) | (sig[:samples_short] > upper)
    axes[i].plot(time[outliers_mask], sig[:samples_short][outliers_mask], 'ro', markersize=2)
    axes[i].set_ylabel('Амплитуда')
    axes[i].set_title(name)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Время (секунды)')
plt.tight_layout()
plt.show()


---
## Этап 4 (продолжение). Скрипичные диаграммы (Violin Plot)
Скрипичные диаграммы показывают полную форму распределения амплитуд — ширину «скрипки» в каждой точке пропорциональна плотности вероятности. В отличие от box plot, они наглядно демонстрируют многомодальность и ширину хвостов: у нормального сигнала — узкая компактная форма, у дефектных — расширенные основания и длинные хвосты, соответствующие ударным импульсам.

In [ ]:
# Визуализация выбросов (boxplot + violin)

import matplotlib.pyplot as plt
import numpy as np

# Скрипичные диаграммы – показывают плотность распределения
fig, ax = plt.subplots(figsize=(12, 6))
parts = ax.violinplot(signals, positions=range(1, len(signals)+1),
                      showmeans=False, showmedians=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.5)
ax.set_xticks(range(1, len(classes)+1))
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.set_ylabel('Амплитуда вибрации')
ax.set_title('Скрипичные диаграммы (Violin plot) – распределение амплитуд')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Этап 4 (продолжение). Гистограммы в логарифмической шкале и QQ-графики
**Гистограммы** строятся с логарифмической шкалой оси Y, что позволяет наглядно увидеть редкие значения в хвостах распределений (подсвечены красным). Вертикальные линии обозначают границы 1-го и 99-го перцентилей.

**QQ-графики** (quantile-quantile plots) сравнивают квантили распределения каждого сигнала с квантилями нормального распределения. Отклонение точек от прямой линии на краях графика указывает на тяжёлые хвосты — характерный признак ударных импульсов дефектных сигналов.

In [ ]:
# Визуализация выбросов на гистограммах

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# 1. Гистограммы с логарифмической шкалой Y и подсветкой хвостов
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (sig, name) in enumerate(zip(signals, classes)):
    # Вычисляем 1-й и 99-й перцентили (границы «нормальных» значений)
    low = np.percentile(sig, 1)
    high = np.percentile(sig, 99)

    # Строим гистограмму
    counts, bins, patches = axes[i].hist(sig, bins=150, alpha=0.7, color='gray', edgecolor='black')

    # Подсвечиваем столбцы, которые попадают в хвосты (выбросы)
    for patch, left, right in zip(patches, bins[:-1], bins[1:]):
        if right < low or left > high:
            patch.set_facecolor('red')
            patch.set_alpha(0.8)

    axes[i].axvline(low, color='red', linestyle='--', alpha=0.7)
    axes[i].axvline(high, color='red', linestyle='--', alpha=0.7)
    axes[i].set_title(name)
    axes[i].set_xlabel('Амплитуда')
    axes[i].set_ylabel('Частота')
    axes[i].grid(alpha=0.3)

    # Логарифмическая шкала Y, чтобы увидеть редкие выбросы
    axes[i].set_yscale('log')
    axes[i].set_ylabel('Частота (log)')

if len(signals) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Гистограммы распределений (логарифмическая шкала, красные хвосты – выбросы)', fontsize=14)
plt.tight_layout()
plt.show()

# QQ-графики
import matplotlib.pyplot as plt
from scipy import stats

indices = [0, 1, 2, 3, 4]  # норма, внутреннее кольцо, шарик, внешнее Centered, внешнее Orthogonal

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, idx in enumerate(indices):
    ax = axes[i]
    stats.probplot(signals[idx], dist="norm", plot=ax)
    ax.set_title(f'QQ-plot: {classes[idx]}')
    ax.grid(alpha=0.3)

if len(indices) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('QQ-графики для проверки «тяжести хвостов» распределений', fontsize=14)
plt.tight_layout()
plt.show()

---
## Этап 5. Анализ диапазонов значений
Строится сводная таблица с характеристиками диапазонов: минимум, максимум, размах и стандартное отклонение для каждого из пяти сигналов. Вычисляются соотношения максимального и минимального размахов и стандартных отклонений.

Дополнительно строятся:
- **KDE-графики** (kernel density estimation) для каждого класса в отдельности — сглаженная оценка плотности распределения амплитуд;
- **Наложенные KDE** для сравнения нормального сигнала с каждым из четырёх дефектных (норма — синий, дефект — красный).

In [ ]:
# Этап 5. Анализ диапазонов значений

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# 1. Таблица со статистиками (min, max, размах, std)
stats_data = []
for name, sig in zip(classes, signals):
    stats_data.append({
        'Состояние': name,
        'Минимум': np.min(sig),
        'Максимум': np.max(sig),
        'Размах': np.max(sig) - np.min(sig),
        'Ст. отклонение': np.std(sig)
    })

df_stats = pd.DataFrame(stats_data)
print("\nТАБЛИЦА 2. ДИАПАЗОНЫ ЗНАЧЕНИЙ СИГНАЛОВ")
print(df_stats.round(4).to_string(index=False))

# Соотношения
max_range = df_stats['Размах'].max()
min_range = df_stats['Размах'].min()
max_std = df_stats['Ст. отклонение'].max()
min_std = df_stats['Ст. отклонение'].min()
print(f"\nСоотношение размахов (max/min): {max_range/min_range:.2f} раза")
print(f"Соотношение стандартных отклонений: {max_std/min_std:.2f} раза")

# 2. Построение кривых плотности на отдельных подграфиках
x_vals = np.linspace(-4, 4, 1000)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
colors = plt.cm.tab10(np.linspace(0, 1, len(signals)))

for i, (name, sig, color) in enumerate(zip(classes, signals, colors)):
    kde = gaussian_kde(sig)
    density = kde(x_vals)
    axes[i].plot(x_vals, density, color=color, linewidth=2)
    axes[i].fill_between(x_vals, density, alpha=0.3, color=color)
    axes[i].set_title(name)
    axes[i].set_xlabel('Амплитуда')
    axes[i].set_ylabel('Плотность')
    axes[i].grid(alpha=0.3)
    axes[i].axvline(np.min(sig), color='gray', linestyle='--', alpha=0.5)
    axes[i].axvline(np.max(sig), color='gray', linestyle='--', alpha=0.5)

if len(signals) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Сглаженные оценки плотности распределения амплитуд вибрационных сигналов', fontsize=14)
plt.tight_layout()
plt.show()

# 3. Дополнительно: наложенные KDE для сравнения с нормой (4 графика)
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 8))
axes2 = axes2.flatten()

normal_sig = signals[0]
normal_name = classes[0]
x_vals2 = np.linspace(-4, 4, 1000)
kde_norm = gaussian_kde(normal_sig)
density_norm = kde_norm(x_vals2)

for i, (name, sig) in enumerate(zip(classes[1:], signals[1:])):
    kde_def = gaussian_kde(sig)
    density_def = kde_def(x_vals2)
    axes2[i].plot(x_vals2, density_norm, color='blue', linewidth=2, label=normal_name)
    axes2[i].plot(x_vals2, density_def, color='red', linewidth=2, label=name)
    axes2[i].fill_between(x_vals2, density_norm, alpha=0.2, color='blue')
    axes2[i].fill_between(x_vals2, density_def, alpha=0.2, color='red')
    axes2[i].set_title(f'{normal_name} vs {name}')
    axes2[i].set_xlabel('Амплитуда')
    axes2[i].set_ylabel('Плотность')
    axes2[i].legend()
    axes2[i].grid(alpha=0.3)

plt.suptitle('Сравнение плотностей распределения (норма — синий, дефект — красный)', fontsize=14)
plt.tight_layout()
plt.show()

---
## Этап 6. Корреляционный анализ
Поскольку сигналы имеют разную длину, перед расчётом корреляции все ряды обрезаются до минимальной длины. Затем вычисляется матрица коэффициентов корреляции Пирсона (5 × 5) и визуализируется в виде тепловой карты.

Дополнительно выводится список пяти наиболее сильных корреляций с указанием направления (положительная/отрицательная) и силы связи.

In [ ]:
# Этап 6. Корреляционный анализ

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Подготовка данных: выравнивание всех сигналов до минимальной длины

min_len = min(len(sig) for sig in signals)
print(f"Минимальная длина сигнала: {min_len} отсчётов")
print("Все сигналы обрезаны до этой длины для корректного расчёта корреляции.\n")

signals_aligned = [sig[:min_len] for sig in signals]
df_corr = pd.DataFrame({name: sig for name, sig in zip(classes, signals_aligned)})

# 2. Расчёт матрицы корреляции Пирсона
corr_matrix = df_corr.corr(method='pearson')

# Вывод матрицы в виде красивой таблицы в консоль
print("МАТРИЦА КОЭФФИЦИЕНТОВ КОРРЕЛЯЦИИ ПИРСОНА (5x5):")
print("="*60)
print(corr_matrix.round(4).to_string())

# 3. Тепловая карта с улучшенной читаемостью
plt.figure(figsize=(10, 8))  # Увеличили размер
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.4f', square=True, linewidths=1,
            annot_kws={'size': 11},  # Увеличили шрифт аннотаций
            cbar_kws={'shrink': 0.8})
plt.title('Тепловая карта корреляции между 5 типами сигналов', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)  # Повернули подписи
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


# 4. Наиболее коррелирующие пары
pairs = []
for i in range(len(classes)):
    for j in range(i+1, len(classes)):
        pairs.append((classes[i], classes[j], corr_matrix.iloc[i, j]))

pairs_sorted = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)

print("\nНАИБОЛЕЕ СИЛЬНЫЕ КОРРЕЛЯЦИИ (по модулю):")
print("="*60)
for a, b, val in pairs_sorted[:5]:
    direction = "положительная" if val > 0 else "отрицательная"
    strength = "очень сильная" if abs(val) >= 0.7 else "сильная" if abs(val) >= 0.5 else "умеренная" if abs(val) >= 0.3 else "слабая"
    print(f"  {a} — {b}: {val:.4f} ({direction}, {strength})")


---
## Этап 7. Поиск и анализ шумов
Для каждого из пяти вибрационных сигналов выполняется декомпозиция методом `seasonal_decompose()` из библиотеки `statsmodels` (аддитивная модель, период = 1 000 отсчётов). Каждый сигнал разбивается на три составляющие: тренд, сезонность и остатки (шум).

По результатам декомпозиции для каждого класса:
- строится четырёхпанельный график (исходный ряд / тренд / сезонность / остатки);
- строится совмещённый график исходного, очищенного сигнала и шума;
- рассчитывается **SNR** (отношение сигнал/шум, дБ) по формуле SNR = 10 · log₁₀(σ²_сигнал / σ²_шум);
- определяется форма распределения остатков по значениям асимметрии и эксцесса.

В конце выводится сводная таблица результатов и рекомендации по фильтрации.

In [ ]:
# Этап 7. Поиск и анализ шумов

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import skew, kurtosis

period = 1000
model = 'additive'

def analyze_signal(signal, name):
    length = len(signal)
    usable = (length // period) * period
    sig = signal[:usable]
    decomp = seasonal_decompose(sig, model=model, period=period)

    trend = decomp.trend
    seas = decomp.seasonal
    resid = decomp.resid
    valid = ~(np.isnan(trend) | np.isnan(seas) | np.isnan(resid))
    trend_clean = trend[valid]
    seas_clean = seas[valid]
    resid_clean = resid[valid]
    observed_clean = sig[valid]

    # Очищенный сигнал (тренд + сезонность)
    signal_clean = trend_clean + seas_clean
    noise = resid_clean

    # Дисперсии и SNR
    var_signal = np.var(signal_clean)
    var_noise = np.var(noise)
    snr = 10 * np.log10(var_signal / var_noise) if var_noise > 0 else np.inf

    # Тренд
    trend_diff = trend_clean[-1] - trend_clean[0]
    if abs(trend_diff) < 0.01 * np.std(signal_clean):
        trend_dir = "стабильный"
        trend_intensity = "слабый"
    elif trend_diff > 0:
        trend_dir = "рост"
        trend_intensity = "умеренный" if abs(trend_diff) < 0.1 else "сильный"
    else:
        trend_dir = "падение"
        trend_intensity = "умеренный" if abs(trend_diff) < 0.1 else "сильный"

    # Сезонность
    seas_amp = (np.max(seas_clean) - np.min(seas_clean)) / 2
    if seas_amp < 0.01 * np.std(signal_clean):
        seas_amp_level = "низкая"
    elif seas_amp < 0.1 * np.std(signal_clean):
        seas_amp_level = "средняя"
    else:
        seas_amp_level = "высокая"

    # Форма шума
    skewness = skew(noise)
    kurt_val = kurtosis(noise)
    if abs(skewness) < 0.5 and abs(kurt_val) < 0.5:
        noise_shape = "нормальное (симметричное)"
    elif abs(skewness) > 1:
        noise_shape = "асимметричное"
    elif kurt_val > 1:
        noise_shape = "с тяжелыми хвостами"
    else:
        noise_shape = "близкое к нормальному"

    # График декомпозиции
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(f'Декомпозиция сигнала: {name}', fontsize=14)
    axes[0].plot(observed_clean, linewidth=0.5, color='black')
    axes[0].set_ylabel('Исходный ряд')
    axes[0].grid(alpha=0.3)
    axes[1].plot(trend_clean, linewidth=0.5, color='red')
    axes[1].set_ylabel('Тренд')
    axes[1].grid(alpha=0.3)
    axes[2].plot(seas_clean, linewidth=0.5, color='green')
    axes[2].set_ylabel('Сезонность')
    axes[2].grid(alpha=0.3)
    axes[3].plot(noise, linewidth=0.5, color='purple')
    axes[3].set_ylabel('Шум (остатки)')
    axes[3].set_xlabel('Время (отсчёты)')
    axes[3].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # График сравнения исходного, очищенного сигнала и шума
    plt.figure(figsize=(12, 6))
    plt.plot(observed_clean, label='Исходный сигнал', linewidth=0.6, color='black', alpha=0.6)
    plt.plot(signal_clean, label='Очищенный (тренд+сезонность)', linewidth=1.2, color='red')
    plt.plot(noise, label='Шум (остатки)', linewidth=0.5, color='blue', alpha=0.5)
    plt.title(f'Сравнение исходного, очищенного сигнала и шума – {name}')
    plt.xlabel('Время (отсчёты)')
    plt.ylabel('Амплитуда')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    return {
        'Сигнал': name,
        'Тренд': f"{trend_dir}, {trend_intensity}",
        'Сезонность': f"период {period}, амплитуда {seas_amp_level}",
        'SNR (дБ)': round(snr, 2),
        'Форма шума': noise_shape,
    }

# Анализ всех сигналов
results = []
for sig, name in zip(signals, classes):
    results.append(analyze_signal(sig, name))

# Сводная таблица
df_results = pd.DataFrame(results)
print("СВОДНЫЕ РЕЗУЛЬТАТЫ ДЕКОМПОЗИЦИИ И ШУМОВОГО АНАЛИЗА")
print(df_results.to_string(index=False))

# Рекомендации по фильтрации
print("РЕКОМЕНДАЦИИ ПО ФИЛЬТРАЦИИ")
for _, row in df_results.iterrows():
    snr = row['SNR (дБ)']
    if snr > 20:
        rec = "Фильтрация не требуется"
    elif snr > 10:
        rec = "Фильтрация не обязательна, можно применить медианный фильтр"
    elif snr > 0:
        rec = "Желательна фильтрация (скользящее среднее или медианный фильтр)"
    else:
        rec = "Обязательна серьёзная фильтрация (например, вейвлет-фильтрация)"
    print(f"{row['Сигнал']}: SNR = {snr:.2f} дБ → {rec}")


---
## Этап 7 (продолжение). Декомпозиция с единым масштабом оси Y
Повторное выполнение декомпозиции, но все четыре подграфика (исходный ряд, тренд, сезонность, остатки) приводятся к **единому масштабу оси Y**. Это позволяет наглядно оценить относительный вклад каждой компоненты: у вибрационных сигналов подшипников тренд и сезонность занимают малую долю от общего размаха, тогда как остатки (ударные импульсы) сопоставимы с исходным сигналом — это и объясняет отрицательные значения SNR.

По итогу выводится сводная таблица с результатами анализа всех пяти сигналов.

In [ ]:
# ============================================
# Этап 7. Поиск и анализ шумов (единый масштаб)
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import skew, kurtosis

period = 1000
model = 'additive'

def analyze_signal(signal, name):
    length = len(signal)
    usable = (length // period) * period
    sig = signal[:usable]
    decomp = seasonal_decompose(sig, model=model, period=period)

    trend = decomp.trend
    seas = decomp.seasonal
    resid = decomp.resid
    valid = ~(np.isnan(trend) | np.isnan(seas) | np.isnan(resid))
    trend_clean = trend[valid]
    seas_clean = seas[valid]
    resid_clean = resid[valid]
    observed_clean = sig[valid]

    # Очищенный сигнал (тренд + сезонность)
    signal_clean = trend_clean + seas_clean
    noise = resid_clean

    # ------------------------------------------------------------
    # 1. Расчёт статистик
    # ------------------------------------------------------------
    var_signal = np.var(signal_clean)
    var_noise = np.var(noise)
    snr = 10 * np.log10(var_signal / var_noise) if var_noise > 0 else np.inf

    trend_diff = trend_clean[-1] - trend_clean[0]
    if abs(trend_diff) < 0.01 * np.std(signal_clean):
        trend_dir = "стабильный"
        trend_intensity = "слабый"
    elif trend_diff > 0:
        trend_dir = "рост"
        trend_intensity = "умеренный" if abs(trend_diff) < 0.1 else "сильный"
    else:
        trend_dir = "падение"
        trend_intensity = "умеренный" if abs(trend_diff) < 0.1 else "сильный"

    seas_amp = (np.max(seas_clean) - np.min(seas_clean)) / 2
    if seas_amp < 0.01 * np.std(signal_clean):
        seas_amp_level = "низкая"
    elif seas_amp < 0.1 * np.std(signal_clean):
        seas_amp_level = "средняя"
    else:
        seas_amp_level = "высокая"

    skewness = skew(noise)
    kurt_val = kurtosis(noise)
    if abs(skewness) < 0.5 and abs(kurt_val) < 0.5:
        noise_shape = "нормальное (симметричное)"
    elif abs(skewness) > 1:
        noise_shape = "асимметричное"
    elif kurt_val > 1:
        noise_shape = "с тяжелыми хвостами"
    else:
        noise_shape = "близкое к нормальному"

    # ------------------------------------------------------------
    # 2. Единый масштаб для всех подграфиков
    # ------------------------------------------------------------
    global_min = min(observed_clean.min(), signal_clean.min(), noise.min())
    global_max = max(observed_clean.max(), signal_clean.max(), noise.max())

    # ------------------------------------------------------------
    # 3. График декомпозиции (единый масштаб)
    # ------------------------------------------------------------
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(f'Декомпозиция сигнала: {name} (единый масштаб Y)', fontsize=14)

    axes[0].plot(observed_clean, linewidth=0.5, color='black')
    axes[0].set_ylabel('Исходный ряд')
    axes[0].set_ylim(global_min, global_max)
    axes[0].grid(alpha=0.3)

    axes[1].plot(trend_clean, linewidth=0.5, color='red')
    axes[1].set_ylabel('Тренд')
    axes[1].set_ylim(global_min, global_max)
    axes[1].grid(alpha=0.3)

    axes[2].plot(seas_clean, linewidth=0.5, color='green')
    axes[2].set_ylabel('Сезонность')
    axes[2].set_ylim(global_min, global_max)
    axes[2].grid(alpha=0.3)

    axes[3].plot(noise, linewidth=0.5, color='purple')
    axes[3].set_ylabel('Шум (остатки)')
    axes[3].set_ylim(global_min, global_max)
    axes[3].set_xlabel('Время (отсчёты)')
    axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # 4. График сравнения исходного, очищенного и шума (единый масштаб)
    # ------------------------------------------------------------
    plt.figure(figsize=(12, 6))
    plt.plot(observed_clean, label='Исходный сигнал', linewidth=0.6, color='black', alpha=0.6)
    plt.plot(signal_clean, label='Очищенный (тренд+сезонность)', linewidth=1.2, color='red')
    plt.plot(noise, label='Шум (остатки)', linewidth=0.5, color='blue', alpha=0.5)
    plt.ylim(global_min, global_max)
    plt.title(f'Сравнение исходного, очищенного сигнала и шума – {name}')
    plt.xlabel('Время (отсчёты)')
    plt.ylabel('Амплитуда')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    return {
        'Сигнал': name,
        'Тренд': f"{trend_dir}, {trend_intensity}",
        'Сезонность': f"период {period}, амплитуда {seas_amp_level}",
        'SNR (дБ)': round(snr, 2),
        'Форма шума': noise_shape,
    }

# ------------------------------------------------------------
# Анализ всех сигналов
# ------------------------------------------------------------
results = []
for sig, name in zip(signals, classes):
    results.append(analyze_signal(sig, name))

# ------------------------------------------------------------
# Сводная таблица
# ------------------------------------------------------------
df_results = pd.DataFrame(results)
print("\nСВОДНЫЕ РЕЗУЛЬТАТЫ ДЕКОМПОЗИЦИИ И ШУМОВОГО АНАЛИЗА")
print(df_results.to_string(index=False))
